# 65. 블라인드 추가검증 실패 진단·남은 라벨 감사
37개를한번평가한뒤실패이유를기술하는사후진단이다. 테스트값으로모델을고치거나재평가성능을독립검증으로발표하지않는다. 센서범위차이는원인증명이아니다. 원본628개RH중이전591+추가37을대조하고압축파일이기존자료복사본인지확인한다.


In [ ]:
from pathlib import Path
import numpy as np,pandas as pd,hashlib,zipfile
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
train=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
train=train[(train['product']=='RG3')&((train.PassOrFail==0)|train.Reason.str.contains('가스',na=False))]
test=pd.read_csv(tables/'rg3_independent37_unblinded_results.csv',parse_dates=['TimeStamp'])
raw=pd.read_csv(root/'data/original/labeled_data.csv')
raw.TimeStamp=pd.to_datetime(raw.TimeStamp)
raw_rg=raw[raw.PART_NAME.str.contains('RG3',na=False)&raw.PART_NAME.str.contains('RH',na=False)]
keys=['EQUIP_CD','PART_NAME','TimeStamp']
seen=pd.read_csv(root/'data/audit/rg3_rh_labeled_with_timestamp.csv',usecols=keys,parse_dates=['TimeStamp'])
all_used=pd.concat([seen,test[keys]],ignore_index=True).drop_duplicates(keys)
remaining=raw_rg[keys].merge(all_used.assign(consumed=True),on=keys,how='left')
remaining=remaining[remaining.consumed.isna()]
assert len(raw_rg)==628 and len(all_used)==628 and len(remaining)==0
cols=pd.read_csv(root/'data/moldset_labeled_rg3.csv',nrows=0).columns.drop(['Unnamed: 0','PassOrFail']).tolist()
test_s=test.merge(raw_rg[keys+cols],on=keys,validate='one_to_one')
ranges=[]
for c in cols:
 tr=train[c];te=test_s[c]
 lo=float(tr.min());hi=float(tr.max())
 ranges.append(dict(sensor=c,train_min=lo,train_max=hi,blind_test_min=float(te.min()),blind_test_max=float(te.max()),outside_train_range=int(((te<lo)|(te>hi)).sum()),test_rows=len(te),gas_values=str(te[test_s.actual_gas.eq(1)].tolist())))
ranges=pd.DataFrame(ranges)
ranges.to_csv(tables/'rg3_independent37_failure_sensor_ranges.csv',index=False,encoding='utf-8-sig')
test_s.to_csv(tables/'rg3_independent37_failure_casebook.csv',index=False,encoding='utf-8-sig')
matches=[]
z=zipfile.ZipFile(root/'data/1. 사출성형기 AI 데이터셋.zip')
for name in z.namelist():
 if name.endswith('.csv'):
  local=root/'data'/Path(name).name
  matches.append(dict(zip_entry=name,local_file=str(local.relative_to(root)),identical_bytes=hashlib.sha256(z.read(name)).hexdigest()==hashlib.sha256(local.read_bytes()).hexdigest()))
pd.DataFrame(matches).to_csv(tables/'rg3_independent_archive_duplicate_audit.csv',index=False,encoding='utf-8-sig')
pd.DataFrame([dict(raw_RG3_RH_rows=len(raw_rg),prior_RG3_RH_rows=len(seen),blind_extra_rows=len(test),remaining_unused_RG3_RH_labels=len(remaining),blind_gas=int(test.actual_gas.eq(1).sum()),blind_normal=int(test.actual_gas.eq(0).sum()))]).to_csv(tables/'rg3_independent_label_exhaustion_audit.csv',index=False,encoding='utf-8-sig')
print('Available independent RG3 RH labels remaining:',len(remaining))
print('Archive repeats existing4files:',all(x['identical_bytes'] for x in matches))
print('Blind gas cases',test_s[test_s.actual_gas.eq(1)][keys+['primary_prediction','reference_prediction']].to_string(index=False))
print('Range differences do NOT prove the cause of failure')
print(ranges.sort_values('outside_train_range',ascending=False).head(8).to_string(index=False))
print('No refitting or retuning performed on consumed37.')
